# Lesson 2.1 — Metrics and the Metric Browser

## Learning Objectives
- Understand AppDynamics' metric model: what a metric is and how it is structured
- Navigate the Metric Browser in the Controller UI
- Construct a metric path for any application, tier, or business transaction metric
- Retrieve metric values via the REST API

**Reference**: [Metrics and Graphs](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/metrics-and-graphs) | [Metric Browser](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/metrics-and-graphs/get-started-with-metric-browser)

## 1. What is a Metric in AppDynamics?

A **metric** is a numeric measurement recorded at regular intervals (typically every minute) by an agent.

Every metric has:
- **A metric path** — a hierarchical address like a file path
- **Time-series values** — one data point per minute
- **A unit** — milliseconds, errors per minute, %, etc.

**Metric path anatomy**:
```
Overall Application Performance|Average Response Time (ms)
│                               │
└─ Category (top-level group)   └─ Metric name

Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)
│                                │                     │           │          │
└─ Category                      └─ Sub-category        └─ Tier     └─ BT name └─ Metric
```

The `|` (pipe) character is the path delimiter.

In [ ]:
# Metric path examples organized by category

metric_examples = [
    # Application-level metrics
    "Overall Application Performance|Average Response Time (ms)",
    "Overall Application Performance|Calls per Minute",
    "Overall Application Performance|Errors per Minute",
    "",
    # Business Transaction metrics
    "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)",
    "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Calls per Minute",
    "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Errors per Minute",
    "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Number of Slow Calls",
    "",
    # JVM metrics (Java agent)
    "JVM|Process CPU Busy (%)",
    "JVM|Heap|Current Usage (MB)",
    "JVM|Garbage Collection|GC Time Spent Per Minute (ms)",
    "",
    # Agent-level machine metrics
    "Hardware Resources|CPU|%Busy",
    "Hardware Resources|Memory|Used (MB)",
    "Hardware Resources|Disks|/|Disk I/O - Reads (per sec)",
]

print("=== Common AppDynamics Metric Paths ===")
for m in metric_examples:
    if m:
        # Count the segments
        segments = m.count('|') + 1
        print(f"  [{segments} segments] {m}")
    else:
        print()

## 2. Navigating the Metric Browser

The **Metric Browser** lets you explore the full metric tree for any application.

### Steps:
1. In the Controller UI, navigate to **Dashboards & Reports → Metric Browser**
   (or use the shortcut in any Application: click the **Metric Browser** link)
2. Select your **Application** from the dropdown
3. Browse the tree on the left: expand categories to find specific metrics
4. Click a metric to see its graph in the right panel
5. Use the time range selector to adjust the window

### Key categories to know:
| Category | Contains |
|----------|----------|
| **Overall Application Performance** | App-level aggregated metrics |
| **Business Transaction Performance** | Per-BT metrics (response time, errors, calls) |
| **JVM** | Java-specific: heap, GC, threads, CPU |
| **CLR** | .NET-specific: garbage collection, threads |
| **Hardware Resources** | Machine agent: CPU, memory, disk, network |
| **Agent** | Agent health and connection metrics |
| **Application Infrastructure Performance** | Per-tier server metrics |

In [ ]:
# REST API: Query metric values
# AppDynamics allows fetching time-series metric data via REST

import os, requests, json
from datetime import datetime

CONTROLLER_URL = os.getenv("APPD_CONTROLLER_URL", "https://your-account.saas.appdynamics.com")
ACCOUNT_NAME   = os.getenv("APPD_ACCOUNT",        "your-account")
USERNAME       = os.getenv("APPD_USER",            "admin")
PASSWORD       = os.getenv("APPD_PASSWORD",        "")
APP_NAME       = os.getenv("APPD_APP_NAME",        "StoreFront")

def get_metric(app_name: str, metric_path: str, duration_in_mins: int = 60) -> list:
    """
    Fetch metric time-series data from AppDynamics.
    
    Returns a list of {startTimeInMillis, value} dicts.
    """
    url = f"{CONTROLLER_URL}/controller/rest/applications/{app_name}/metric-data"
    params = {
        "metric-path": metric_path,
        "time-range-type": "BEFORE_NOW",
        "duration-in-mins": duration_in_mins,
        "output": "JSON"
    }
    auth = (f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD)
    resp = requests.get(url, auth=auth, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if data:
        return data[0].get("metricValues", [])
    return []

print("get_metric() function defined.")
print(f"Will query application: {APP_NAME}")

In [ ]:
# Fetch and display the last 30 minutes of overall application response time

try:
    metric_path = "Overall Application Performance|Average Response Time (ms)"
    values = get_metric(APP_NAME, metric_path, duration_in_mins=30)
    
    if values:
        print(f"Metric: {metric_path}")
        print(f"{'Timestamp':<25} {'Value (ms)':>10}")
        print("-" * 38)
        for v in values[-10:]:  # last 10 data points
            ts = datetime.fromtimestamp(v['startTimeInMillis'] / 1000).strftime('%Y-%m-%d %H:%M:%S')
            print(f"{ts:<25} {v['value']:>10.1f}")
    else:
        print("No metric data returned.")

except Exception as e:
    print(f"[Not connected] Showing simulated data instead...")
    print()
    # Simulated output
    import random, time
    random.seed(99)
    print(f"Metric: Overall Application Performance|Average Response Time (ms)")
    print(f"{'Timestamp':<25} {'Value (ms)':>10}")
    print("-" * 38)
    base_ts = int(time.time()) - 600  # 10 mins ago
    for i in range(10):
        ts = datetime.fromtimestamp(base_ts + i * 60).strftime('%Y-%m-%d %H:%M:%S')
        val = random.gauss(285, 40)
        print(f"{ts:<25} {val:>10.1f}")

## 3. Wildcards in Metric Paths

The AppDynamics REST API supports wildcards (`*`) in metric paths to query multiple metrics at once:

```
# All BTs in WebFrontend tier:
Business Transaction Performance|Business Transactions|WebFrontend|*|Average Response Time (ms)

# Average response time across ALL tiers:
Business Transaction Performance|Business Transactions|*|*|Average Response Time (ms)
```

This is useful for bulk exports and cross-tier comparisons.

In [ ]:
# Fetch metrics for ALL BTs in the WebFrontend tier using a wildcard

TIER_NAME = "WebFrontend"  # TODO: change to your tier name

wildcard_path = f"Business Transaction Performance|Business Transactions|{TIER_NAME}|*|Calls per Minute"

try:
    url = f"{CONTROLLER_URL}/controller/rest/applications/{APP_NAME}/metric-data"
    params = {
        "metric-path": wildcard_path,
        "time-range-type": "BEFORE_NOW",
        "duration-in-mins": 15,
        "rollup": True,  # return a single aggregated value per metric
        "output": "JSON"
    }
    auth = (f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD)
    resp = requests.get(url, auth=auth, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    
    print(f"BT call rates in tier '{TIER_NAME}' (last 15 mins):")
    for item in data:
        path = item.get('metricPath', '?')
        bt_name = path.split('|')[-2] if '|' in path else path
        val = item['metricValues'][0]['value'] if item.get('metricValues') else 0
        print(f"  {bt_name:<30}: {val:.1f} calls/min")

except Exception as e:
    print(f"[Simulated output]")
    sample = {"/checkout": 12.4, "/search": 87.1, "/cart/add": 34.2, "/cart/view": 44.8}
    for bt, cpm in sample.items():
        print(f"  {bt:<30}: {cpm:.1f} calls/min")

In [ ]:
# EXERCISE 2.1: Build a metric path from components
#
# TASK: Write a function that constructs a Business Transaction metric path
# from individual components.
#
# Example call:
#   build_bt_metric_path("WebFrontend", "/checkout", "Average Response Time (ms)")
#   → "Business Transaction Performance|Business Transactions|WebFrontend|/checkout|Average Response Time (ms)"

def build_bt_metric_path(tier_name: str, bt_name: str, metric_name: str) -> str:
    """
    Construct a fully qualified Business Transaction metric path.
    """
    # TODO: implement
    pass

# Test cases
test_cases = [
    ("WebFrontend",  "/checkout",   "Average Response Time (ms)"),
    ("OrderService", "/api/orders", "Errors per Minute"),
    ("PaymentSvc",   "/pay",        "Number of Very Slow Calls"),
]

for tier, bt, metric in test_cases:
    result = build_bt_metric_path(tier, bt, metric)
    print(f"Result: {result}")

## Summary

- AppDynamics metrics are identified by **metric paths** — hierarchical strings using `|` as delimiter
- Categories include: Overall Application Performance, Business Transaction Performance, JVM, Hardware Resources
- The **Metric Browser** lets you explore and graph any metric in the UI
- The REST API endpoint `/metric-data` retrieves time-series values; supports **wildcard** paths
- Metric data rolls up every minute — you'll see one data point per minute

---

**Next**: [Lesson 2.2 — Business Transaction Performance and Health](lesson-2.2-bt-performance-health.ipynb)